# 4. Comparison with EIP-7999

Assembles the report's main comparison table, complete five-series comparison, paired gains, sensitivity tables, and remaining appendix figures from the simulations in Notebooks 1–3 and the upstream EIP-7999 workflow.

The main table selects one configuration per design family by highest mean delivered execution across propagation allocations. Historically anchored EIP-7999 is selected within its eligibility rule. Keep physical state growth and hard-limit frequency beside execution: the configurations have different capacity use and operating pressure.

This notebook performs comparison and presentation calculations. Its inputs are generated by the preceding notebooks.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/shared_fee/replay.py").is_file())
for directory in (ROOT / "src", ROOT / "scripts", ROOT / "scripts/shared_fee"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from publication_workflow import (
    DATA, REPORT, LABELS, baseline_anchor, read_table, require_files,
    run_stage, shared_display, source_snapshot, verify_sources,
)
DATA.mkdir(parents=True, exist_ok=True)
(ROOT / "plots").mkdir(exist_ok=True)
REUSE = os.environ.get("ONE_DIMENSIONAL_REUSE_OUTPUTS", "0") == "1"
REFRESH_XATU = os.environ.get("ONE_DIMENSIONAL_REFRESH_XATU", "0") == "1"
before = source_snapshot()
pd.set_option("display.max_columns", 24)
print("Repository:", ROOT)
print("Reuse generated outputs:", REUSE, "| Refresh Xatu inputs:", REFRESH_XATU)

## Main comparison: one configuration per family

All five series use the central 35-day vector and unrestricted state demand. Selection maximizes the mean across the same 32 paths. One-dimensional fees are shared; EIP-7999 fees are ordered execution / data / state.

The fee exposure column is a modeled burn measure for EIP-7999 and a counter-weighted charge proxy for one-dimensional mechanisms. For EIP-8372 it uses included **raw** state gas. Transaction-floor maxima and refunds prevent interpreting that proxy as exact burned ETH.

In [ ]:
comparison = read_table("shared_fee_elasticity_comparison.csv")
central = comparison[comparison.window_days.eq(35) & comparison.available].copy()
normalized = read_table("eip8372/normalized_outcomes.csv")
normalized = normalized[normalized.window_days.eq(35)].copy()
normalized["equilibrium_execution_fee_wei"] = normalized.equilibrium_fee_wei
all_central = pd.concat([central, normalized], ignore_index=True)
assert len(all_central) == 25 and all_central.groupby("benchmark").size().eq(5).all()
best = all_central.loc[all_central.groupby("benchmark").metered_execution_gas.idxmax()]
best = best.set_index("benchmark").loc[list(LABELS)].reset_index()

def fees(row):
    if row.benchmark in ("maximum", "balanced"):
        return " / ".join(f"{row[f'equilibrium_{resource}_fee_wei']:,.2f}"
                          for resource in ("execution", "data", "state"))
    return f"{row.equilibrium_execution_fee_wei:,.2f}"

summary = pd.DataFrame({
    "Design": best.benchmark.map(LABELS),
    "Configuration": [
        f"{row.propagation_time_s:.1f}s, " + (
            row.configuration if row.benchmark in ("maximum", "balanced")
            else f"{row.shared_limit / 1e6:g}M limit")
        for row in best.itertuples()
    ],
    "Equilibrium fee(s) (wei)": best.apply(fees, axis=1),
    "Mean execution (M)": best.metered_execution_gas / 1e6,
    "State growth (GiB/year)": best.annualized_state_growth_gib,
    "Blocks at limit (%)": 100 * best.hard_limit_fraction,
    "Mean burn / charge proxy (ETH/block)": best.base_fee_exposure_proxy_eth_per_block,
})
display(summary)
summary.to_csv(DATA / "publication_comparison_summary.csv", index=False)
all_central.to_csv(DATA / "publication_comparison_all_allocations.csv", index=False)
np.testing.assert_allclose(best.metered_execution_gas / 1e6,
                           [92.6, 70.2, 177.9, 223.0, 272.6], rtol=0, atol=0.05)
assert best.propagation_time_s.tolist() == [4., 3.5, 3.5, 4., 4.5]

## Comparison at each propagation allocation

These 25 rows reproduce the full unrestricted comparison appendix. The figure retains the same propagation time when comparing the five families, complementing the globally selected table above.

In [ ]:
display(all_central[["benchmark", "propagation_time_s", "configuration",
                     "equilibrium_execution_fee_wei", "equilibrium_data_fee_wei",
                     "equilibrium_state_fee_wei", "metered_execution_gas",
                     "data_or_floor_gas", "annualized_state_growth_gib",
                     "hard_limit_fraction", "execution_price_variation",
                     "base_fee_exposure_proxy_eth_per_block"]]
        .sort_values(["propagation_time_s", "benchmark"]))
import make_figures
import make_eip8372_figures
make_figures.paired_dot_comparison()
display(Image(filename=str(ROOT / "plots/shared_fee_comparison_execution_paired_dot.png")))

## Frozen-design elasticity gains

Subtract EIP-8372 execution from EIP-7999 execution **within the same path, elasticity vector, and propagation allocation**, then average the 32 differences. Constants and target pairs retain their central calibration. The historical-selection label identifies the central design; it need not remain eligible under another elasticity vector.

The main table reports the range of mean gains across five propagation allocations. Weekly p05–p95 values measure dispersion across simulated weeks, not confidence intervals for the mean.

In [ ]:
gains = read_table("eip8372/paired_gains.csv")
gain_paths = read_table("eip8372/paired_gain_paths.csv")
assert len(gains) == 40 and len(gain_paths) == 40 * 32
keys = ["benchmark_7999", "window_days", "propagation_time_s"]
computed = gain_paths.groupby(keys).execution_gain.mean().sort_index()
reported = gains.set_index(["benchmark", "window_days", "propagation_time_s"]).mean_execution_gain.sort_index()
np.testing.assert_allclose(computed, reported, rtol=1e-12)
assert gain_paths.groupby(keys).replication.nunique().eq(32).all()
ranges = gains.assign(mean_execution_gain_M=gains.mean_execution_gain / 1e6).groupby(
    ["window_days", "benchmark"]).mean_execution_gain_M.agg(["min", "max"])
display(ranges)
display(gains.sort_values(["window_days", "propagation_time_s", "benchmark"]))
assert (gains.mean_execution_gain > 0).all()
make_eip8372_figures.elasticity_figure()
make_eip8372_figures.three_mechanism_elasticity_figure()
display(Image(filename=str(ROOT / "plots/shared_fee_elasticity_execution_state.png")))
display(Image(filename=str(ROOT / "plots/shared_fee_eip8372_frozen_elasticities.png")))

## Broader reselected and frozen-central appendix tables

The broader elasticity experiment also reselects configurations within each vector. Its EIP-7999 gain reference is floor-adjusted + EIP-8368, whereas the main fixed-design comparison above uses floor-adjusted + EIP-8372. Missing eligible configurations are displayed explicitly. Sampling intervals for the mean hold the selected design fixed and omit selection, elasticity-estimation, and shock-reconstruction uncertainty.

In [ ]:
from elasticity_report_tables import report_tables
for name, table in report_tables().items():
    display(Markdown("### " + name.replace("_", " ").title() + "\n\n" + table))
make_figures.elasticity_robustness_figures()
make_figures.elasticity_state_growth_figure()
make_figures.elasticity_equilibrium_fee_figure()
for name in ["shared_fee_elasticity_performance.png", "shared_fee_elasticity_state_growth.png",
             "shared_fee_elasticity_equilibrium_fee.png", "shared_fee_elasticity_execution_gains.png"]:
    display(Image(filename=str(ROOT / "plots" / name)))

## State-tail comparison and state-pulse transmission

The three-second state-cap table joins the two earlier mechanisms and EIP-8372. Its paired gains use the fixed unrestricted EIP-7999 references. Separately, the pulse table compares each disturbance with an identical no-pulse path, so it isolates the transmission channel within each design.

In [ ]:
tail = read_table("eip8372/state_tail/three_second_comparison.csv")
display(tail)
caps = ["unrestricted", "1.5x", "2x"]
display((tail[tail.state_demand_cap_label.isin(caps)].pivot(
    index="benchmark", columns="state_demand_cap_label", values="metered_execution_gas")
    .reindex(index=["proposal_faithful", "fully_optimized", "normalized_state"],
             columns=caps) / 1e6).rename(index=LABELS))
display(read_table("eip8372/state_tail/paired_gains.csv"))
display(read_table("eip8372/stress_outcomes.csv"))

## Verify report figures and provenance

Check that the report's figure links resolve and every simulation manifest retains the same workload hash. Large source panels still require the documented upstream Xatu/RPC access.

In [ ]:
import re
figures = re.findall(r"!\[[^\]]*\]\(([^)]+)\)", REPORT.read_text())
assert figures, "No figure references found in the report"
require_files([(REPORT.parent / ref).resolve() for ref in figures])
workload_hash = read_table("shared_fee_factorial_manifest.csv").iloc[0].workload_sha256
for name in ["shared_fee_elasticity_comparison_manifest.json",
             "shared_fee_elasticity_robustness_manifest.json", "eip8372/manifest.json",
             "eip8372/state_utilization_diagnostic_manifest.json", "eip8372/state_tail/manifest.json"]:
    manifest = json.loads((DATA / name).read_text())
    assert manifest["workload_sha256"] == workload_hash, name
print(f"All {len(figures)} report figures resolve; simulation manifests share workload {workload_hash}.")
verify_sources(before)